# DVRP — Framework GNN + RL pour le routage dynamique incertain et sensible au risque

**Architecture** : encoder GNN edge-aware (TransformerConv, résiduel, contexte de zone, Edge Gating) → encoder véhicule → Attention Decoder (glimpse multi-tête + pointer) → biais de risque explicite → REINFORCE + Learned Critic, scheduler event-driven.

Organisation :
1. Imports / configuration
2. Environnement
3. Préparation des observations
4. Encoder GNN
5. Encoder véhicule
6. Decoder attention
7. Policy
8. Reward
9. Rollout event-driven
10. REINFORCE + critic
11. Training
12. Exécution de l'entraînement
13. Tests, diagnostics et comparaison

Les sections 1 à 11 ne contiennent que des définitions (classes/fonctions), aucune exécution.


## 1. Imports / Configuration

In [ ]:

import json, math, os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Batch
from torch_geometric.nn import TransformerConv
from dataclasses import dataclass
from collections import namedtuple

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# CONFIGURATION CENTRALISÉE
# ============================================================
CONFIG = {
    # --- Architecture : encoder ---
    "node_dim": 15,
    "edge_dim": 4,
    "embed_dim": 128,
    "encoder_layers": 1,
    "encoder_heads": 4,
    "encoder_n_zones": 25,  # = zone_grid 5x5 (scenario_params.json), utilise pour le contexte de zone
    "dropedge_rate": 0.1,
    "risk_scale_K": 400,             # facteur d'echelle fixe applique au biais de risque sur les logits

    # --- Architecture : vehicle encoder ---
    "vehicle_dim": 6,
    "vehicle_hidden": 64,

    # --- Architecture : decoder ---
    "decoder": "attention",
    "decoder_heads": 8,              # nb têtes pour l'attention "glimpse" (Kool et al. utilise 8)
    "logit_clip": 10.0,              # clipping tanh sur les logits finaux (stabilise l'entraînement, Kool et al.)

    # --- RL ---
    "baseline_type": "critic",

    # --- Reward : coefficients (hyperparamètres expérimentaux, non calibrés) ---
    "reward_alpha": 1.15,             # travel time
    "reward_beta":  1.0,             # lateness
    "reward_gamma": 5.0,             # rejected orders (terminal)
    "service_bonus": 0.10,           # bonus immediat par client reellement servi
    "reward_delta": 2.0,             # overtime (terminal)
    "reward_eta":   0.5,             # risk (sigma, pénalisation linéaire)
    "day_length": 480.0,

    # --- Training ---
    "lr_policy": 1e-4,
    "lr_baseline": 1e-3,
    "discount_gamma": 0.99,
    "batch_episodes": 2,             # nb épisodes accumulés avant un pas de gradient
    "grad_clip_norm": 1.0,
    "normalize_returns": True,
    "n_training_episodes": 1000,
    "log_every": 20,

    # --- Protocole d'évaluation ---
    # train_seed_offset : les seeds d'entrainement sont generees comme
    # train_seed_offset + n_episodes_done + i -- croissantes et jamais
    # reutilisees. Marge de securite large pour ne jamais chevaucher
    # validation_seeds (50-99) ni test_seeds (0-49).
    "train_seed_offset": 100000,
    "validation_seeds": list(range(50, 100)),
    "test_seeds": list(range(100000000, 100000050)),

    # --- Régularisation ---
    "dropout": 0.0,
    "use_residual": False,
    "weight_decay": 0.0,
}

Transition = namedtuple("Transition", ["log_prob", "value_est", "reward"])


## 2. Environnement

In [ ]:

CLIENT_PENDING   = "PENDING"
CLIENT_ACTIVE    = "ACTIVE"
CLIENT_SERVED    = "SERVED"
CLIENT_CANCELLED = "CANCELLED"
DEPOT_ID = "D0"

def load_params(path="config/scenario_params.json"):
    abs_path = os.path.abspath(path)
    if not os.path.exists(abs_path):
        raise FileNotFoundError(f"scenario_params.json introuvable : {abs_path}\ncwd : {os.getcwd()}")
    if os.path.getsize(abs_path) == 0:
        raise ValueError(f"{abs_path} est VIDE (0 octet).")
    with open(abs_path, "r", encoding="utf-8") as f:
        return json.load(f)

def load_demand_pool(path="config/demand_pool_solomon.csv"):
    abs_path = os.path.abspath(path)
    if not os.path.exists(abs_path):
        raise FileNotFoundError(f"Pool Solomon introuvable : {abs_path}\ncwd : {os.getcwd()}")
    df = pd.read_csv(abs_path)
    if "demand" not in df.columns:
        raise ValueError(f"Colonne 'demand' absente. Colonnes : {list(df.columns)}")
    return df["demand"].to_numpy()

def normalize_xy(x, y, bounds):
    lo, hi = bounds
    return (x - lo) / (hi - lo), (y - lo) / (hi - lo)

def get_zone_id(x, y, bounds, grid):
    x_norm, y_norm = normalize_xy(x, y, bounds)
    n_cols, n_rows = grid["n_cols"], grid["n_rows"]
    col = min(int(x_norm * n_cols), n_cols - 1)
    row = min(int(y_norm * n_rows), n_rows - 1)
    return row * n_cols + col

def zone_centrality_table(grid):
    n_cols, n_rows = grid["n_cols"], grid["n_rows"]
    max_dist = math.sqrt(0.5**2 + 0.5**2)
    table = {}
    for row in range(n_rows):
        for col in range(n_cols):
            zid = row * n_cols + col
            cx, cy = (col + 0.5) / n_cols, (row + 0.5) / n_rows
            dist = math.sqrt((cx - 0.5)**2 + (cy - 0.5)**2)
            table[zid] = dist / max_dist
    return table

def traffic_base_by_zone(grid, alpha):
    centrality = zone_centrality_table(grid)
    return {z: 1 + alpha * (1 - d) for z, d in centrality.items()}

def sample_tw_start(rng, n, bounds, median_target):
    lo, hi = bounds
    lam = math.log(2) / median_target
    u = rng.uniform(0, 1, size=n)
    x = -np.log(1 - u) / lam
    return np.clip(x, lo, hi - 1e-6)

def sample_tw_width(rng, n, components):
    values  = [c["value"]  for c in components]
    weights = [c["weight"] for c in components]
    return rng.choice(values, size=n, p=weights).astype(float)

def traffic_factor(zone_id, t, traffic_base, hour_bands, noise_std, rng):
    base = traffic_base[zone_id]
    mult = 1.0
    for band in hour_bands:
        if band["t_start"] <= t < band["t_end"]:
            mult = band["multiplier"]
            break
    noise = 1 + rng.normal(0, noise_std)
    return base * mult * max(noise, 0.1)

def _get_hour_multiplier(t, hour_bands):
    for band in hour_bands:
        if band["t_start"] <= t < band["t_end"]:
            return band["multiplier"]
    return 1.0

def _gen_depot(rng, params):
    bounds = params["client_attrs"]["coordinates_generation"]["bounds"]
    x = rng.uniform(bounds[0], bounds[1])
    y = rng.uniform(bounds[0], bounds[1])
    zone_id = get_zone_id(x, y, bounds, params["traffic_and_space"]["zone_grid"])
    return {
        "id": "D0", "type": "depot", "x": x, "y": y, "zone_id": zone_id,
        "status": "DEPOT",
        "demand": np.nan, "tw_start": np.nan, "tw_end": np.nan,
        "service_time": np.nan, "capacity": np.inf,
        "revealed_at": 0.0,
    }

def _gen_clients(rng, params, demand_pool, n_clients, status, id_offset):
    bounds   = params["client_attrs"]["coordinates_generation"]["bounds"]
    grid     = params["traffic_and_space"]["zone_grid"]
    horizon  = params["horizon"]["day_length"]
    svc_time = params["client_attrs"]["service_time"]["value"]
    tw_comps = params["client_attrs"]["tw_width"]["components"]
    tw_cfg   = params["client_attrs"]["tw_start"]

    xs        = rng.uniform(bounds[0], bounds[1], size=n_clients)
    ys        = rng.uniform(bounds[0], bounds[1], size=n_clients)
    demands   = rng.choice(demand_pool, size=n_clients)
    tw_widths = sample_tw_width(rng, n_clients, tw_comps)
    tw_starts = sample_tw_start(rng, n_clients, tw_cfg["bounds"], tw_cfg["median_target"])
    tw_ends   = np.minimum(tw_starts + tw_widths, horizon)

    revealed_at = 0.0 if status == CLIENT_ACTIVE else np.nan

    rows = []
    for i in range(n_clients):
        zone_id = get_zone_id(xs[i], ys[i], bounds, grid)
        rows.append({
            "id": f"C{id_offset + i}", "type": "client",
            "x": xs[i], "y": ys[i], "zone_id": zone_id,
            "status": status,
            "demand": demands[i],
            "tw_start": tw_starts[i], "tw_end": tw_ends[i],
            "service_time": svc_time, "capacity": np.nan,
            "revealed_at": revealed_at,
        })
    return rows

def _gen_vehicles(rng, params, depot_id, depot_x, depot_y):
    n_vehicles  = params["entities"]["n_vehicles_total"]
    capacity    = params["vehicle_attrs"]["capacity_reference_solomon"]["value"]
    shift_start = params["vehicle_attrs"]["shift_start"]
    shift_end   = params["vehicle_attrs"]["shift_end"]
    bounds      = params["client_attrs"]["coordinates_generation"]["bounds"]
    grid        = params["traffic_and_space"]["zone_grid"]

    rows = []
    for vid in range(n_vehicles):
        rows.append({
            "id": f"V{vid}", "depot_id": depot_id,
            "capacity": capacity, "remaining_capacity": capacity,
            "current_x": depot_x, "current_y": depot_y,
            "current_zone_id": get_zone_id(depot_x, depot_y, bounds, grid),
            "shift_start": shift_start, "shift_end": shift_end,
            "remaining_shift_time": float(shift_end - shift_start),
            "total_distance_km": 0.0,
            "total_travel_time": 0.0,
            "overtime": 0.0,
            "total_lateness": 0.0,
            "n_deliveries": 0,
            "n_depot_returns": 0,
        })
    return rows

def build_matrices(nodes_df, params):
    scaling     = params["traffic_and_space"]["scaling_factor_spatial"]["value"]
    road_factor = params["traffic_and_space"]["road_factor"]["value"]
    speed_kmh   = params["traffic_and_space"]["vehicle_speed_kmh"]["value"]
    noise_std   = params["traffic_and_space"]["hour_multiplier"]["traffic_noise_std"]["value"]
    alpha       = params["traffic_and_space"]["traffic_base_centrality"]["alpha_centrality"]
    hour_bands  = params["traffic_and_space"]["hour_multiplier"]["bands"]
    grid        = params["traffic_and_space"]["zone_grid"]

    coords     = nodes_df[["x", "y"]].to_numpy()
    diff       = coords[:, None, :] - coords[None, :, :]
    dist_units = np.sqrt((diff**2).sum(axis=-1))

    dist_km_road  = dist_units * scaling * road_factor
    time_min_base = dist_km_road / speed_kmh * 60.0

    centrality = zone_centrality_table(grid)
    hour_mult_t0 = _get_hour_multiplier(0.0, hour_bands)
    n = len(nodes_df)
    uncertainty_mat = np.zeros((n, n), dtype=np.float32)
    for i in range(n):
        zone_i   = int(nodes_df.iloc[i]["zone_id"])
        tb_i     = 1 + alpha * (1 - centrality[zone_i])
        sigma_i  = noise_std * tb_i * hour_mult_t0
        uncertainty_mat[i, :] = sigma_i * time_min_base[i, :]

    return dist_km_road, time_min_base, uncertainty_mat

@dataclass
class Instance:
    nodes_df:        pd.DataFrame
    vehicles_df:     pd.DataFrame
    dist_matrix_km:  np.ndarray
    time_matrix_min: np.ndarray
    uncertainty_matrix: np.ndarray
    traffic_base:    dict
    params:          dict
    seed:            int

def generate_instance(seed,
                      params_path="config/scenario_params.json",
                      demand_pool_path="config/demand_pool_solomon.csv"):
    params      = load_params(params_path)
    demand_pool = load_demand_pool(demand_pool_path)
    rng         = np.random.default_rng(seed)

    depot_row    = _gen_depot(rng, params)
    n_active     = params["entities"]["n_clients_active_initial"]
    n_reserve    = params["entities"]["n_clients_reserve"]
    active_rows  = _gen_clients(rng, params, demand_pool, n_active,  CLIENT_ACTIVE,  0)
    reserve_rows = _gen_clients(rng, params, demand_pool, n_reserve, CLIENT_PENDING, n_active)

    nodes_df    = pd.DataFrame([depot_row] + active_rows + reserve_rows)
    vehicles_df = pd.DataFrame(
        _gen_vehicles(rng, params, depot_row["id"], depot_row["x"], depot_row["y"])
    )
    dist_km, time_min, uncert = build_matrices(nodes_df, params)
    tb = traffic_base_by_zone(
        params["traffic_and_space"]["zone_grid"],
        params["traffic_and_space"]["traffic_base_centrality"]["alpha_centrality"],
    )
    return Instance(
        nodes_df=nodes_df, vehicles_df=vehicles_df,
        dist_matrix_km=dist_km, time_matrix_min=time_min,
        uncertainty_matrix=uncert,
        traffic_base=tb, params=params, seed=seed,
    )


class VRPDynamicEnv:
    """Environnement VRP dynamique. reset()/step() ; voir get_observation()
    pour le format consommé par la policy GNN."""

    def __init__(self,
                 params_path="config/scenario_params.json",
                 demand_pool_path="config/demand_pool_solomon.csv"):
        self.params_path      = params_path
        self.demand_pool_path = demand_pool_path
        self.instance = None

    def reset(self, seed):
        self.instance = generate_instance(seed, self.params_path, self.demand_pool_path)
        self.rng = np.random.default_rng(seed + 999983)

        p = self.instance.params
        self.day_length   = p["horizon"]["day_length"]
        self.bounds       = p["client_attrs"]["coordinates_generation"]["bounds"]
        self.grid         = p["traffic_and_space"]["zone_grid"]
        self.traffic_base = self.instance.traffic_base
        self.hour_bands   = p["traffic_and_space"]["hour_multiplier"]["bands"]
        self.noise_std    = p["traffic_and_space"]["hour_multiplier"]["traffic_noise_std"]["value"]
        self.alpha        = p["traffic_and_space"]["traffic_base_centrality"]["alpha_centrality"]
        self.lambda_step  = p["dynamics"]["poisson_rate_per_step"]
        self.cancel_prob  = p["dynamics"]["cancel_prob_per_step"]

        self.nodes    = self.instance.nodes_df.reset_index(drop=True).copy()
        self.vehicles = self.instance.vehicles_df.reset_index(drop=True).copy()
        for col in ["remaining_capacity", "current_x", "current_y",
                    "remaining_shift_time", "total_distance_km",
                    "total_travel_time", "overtime", "total_lateness"]:
            self.vehicles[col] = self.vehicles[col].astype(float)

        self.dist_km  = self.instance.dist_matrix_km
        self.time_min = self.instance.time_matrix_min
        self.uncert   = self.instance.uncertainty_matrix

        self.node_index = {nid: i for i, nid in enumerate(self.nodes["id"])}
        self.vehicle_current_time = {vid: 0.0 for vid in self.vehicles["id"]}
        self.vehicle_position_idx = {vid: self.node_index[DEPOT_ID]
                                     for vid in self.vehicles["id"]}
        self.step_counter     = 0
        self.done             = False
        self._invalid_attempts = 0

        # Fenetre d'exposition individuelle + plafond global d'annulations
        self.activation_step = np.where(
            self.nodes["status"].to_numpy() == CLIENT_ACTIVE, 0, -1
        )
        self.cancel_window_steps     = 10   # un client ne peut plus etre annule apres 10 pas d'activite
        self.max_cancellations_total = 30   # plafond global d'annulations sur tout l'episode
        self.n_cancellations_total   = 0

        return self._get_state(), self.get_observation()

    def _sigma(self, zone_i, t):
        tb   = self.traffic_base[zone_i]
        mult = 1.0
        for band in self.hour_bands:
            if band["t_start"] <= t < band["t_end"]:
                mult = band["multiplier"]
                break
        return self.noise_std * tb * mult

    def step(self, vehicle_id, destination_id):
        if self.done:
            raise RuntimeError("Episode terminé — appeler reset().")

        info = {"valid": True, "reason": None, "action_type": None,
                "travel_time_realized": None, "sigma_used": None,
                "lateness": 0.0, "service_completed": False}

        v_idx = self.vehicles.index[self.vehicles["id"] == vehicle_id][0]
        v_row = self.vehicles.iloc[v_idx]

        origin_idx  = self.vehicle_position_idx[vehicle_id]
        t_now       = self.vehicle_current_time[vehicle_id]
        origin_zone = int(self.nodes.loc[origin_idx, "zone_id"])

        tf = traffic_factor(origin_zone, t_now, self.traffic_base,
                            self.hour_bands, self.noise_std, self.rng)

        if destination_id == DEPOT_ID:
            depot_idx    = self.node_index[DEPOT_ID]
            sigma_val    = self._sigma(origin_zone, t_now)
            mu_travel    = self.time_min[origin_idx, depot_idx] * tf
            eps          = self.rng.normal(0, sigma_val * self.time_min[origin_idx, depot_idx])
            travel_time  = max(mu_travel + eps, 0.0)
            dist_km      = self.dist_km[origin_idx, depot_idx]
            arrival_time = t_now + travel_time

            if arrival_time > v_row["shift_end"]:
                info.update(valid=False, reason="retour dépôt dépasse fin de shift")
                self._invalid_attempts += 1
                return self._get_state(), self.get_observation(), info

            self.vehicles.at[v_idx, "current_x"]            = self.nodes.loc[depot_idx, "x"]
            self.vehicles.at[v_idx, "current_y"]            = self.nodes.loc[depot_idx, "y"]
            self.vehicles.at[v_idx, "current_zone_id"]      = int(self.nodes.loc[depot_idx, "zone_id"])
            self.vehicles.at[v_idx, "remaining_capacity"]   = float(v_row["capacity"])
            self.vehicles.at[v_idx, "remaining_shift_time"] = v_row["shift_end"] - arrival_time
            self.vehicles.at[v_idx, "total_distance_km"]   += dist_km
            self.vehicles.at[v_idx, "total_travel_time"]   += travel_time
            self.vehicles.at[v_idx, "n_depot_returns"]     += 1
            self.vehicle_current_time[vehicle_id] = arrival_time
            self.vehicle_position_idx[vehicle_id] = depot_idx
            info["action_type"] = "depot_return"
            info["travel_time_realized"] = round(travel_time, 3)
            info["sigma_used"] = round(sigma_val, 4)

        else:
            c_idx = self.node_index.get(destination_id)
            if c_idx is None:
                info.update(valid=False, reason="destination inconnue")
                self._invalid_attempts += 1
                return self._get_state(), self.get_observation(), info

            c_row  = self.nodes.iloc[c_idx]
            status = c_row["status"]

            if status != CLIENT_ACTIVE:
                info.update(valid=False, reason=f"client status={status}")
                self._invalid_attempts += 1
                return self._get_state(), self.get_observation(), info
            if c_row["demand"] > v_row["remaining_capacity"]:
                info.update(valid=False, reason="capacité insuffisante")
                self._invalid_attempts += 1
                return self._get_state(), self.get_observation(), info

            sigma_val    = self._sigma(origin_zone, t_now)
            mu_travel    = self.time_min[origin_idx, c_idx] * tf
            eps          = self.rng.normal(0, sigma_val * self.time_min[origin_idx, c_idx])
            travel_time  = max(mu_travel + eps, 0.0)
            dist_km      = self.dist_km[origin_idx, c_idx]
            arrival_time = t_now + travel_time
            service_start = max(arrival_time, c_row["tw_start"])

            lateness = max(0.0, service_start - c_row["tw_end"])
            service_end = service_start + c_row["service_time"]
            overtime = max(0.0, service_end - v_row["shift_end"])

            self.nodes.at[c_idx, "status"]                  = CLIENT_SERVED
            info["service_completed"] = True   # bonus attribue ici uniquement -- livraison reellement aboutie
            self.vehicles.at[v_idx, "remaining_capacity"]  -= c_row["demand"]
            self.vehicles.at[v_idx, "current_x"]            = c_row["x"]
            self.vehicles.at[v_idx, "current_y"]            = c_row["y"]
            self.vehicles.at[v_idx, "current_zone_id"]      = int(c_row["zone_id"])
            self.vehicles.at[v_idx, "remaining_shift_time"] = v_row["shift_end"] - service_end
            self.vehicles.at[v_idx, "total_distance_km"]   += dist_km
            self.vehicles.at[v_idx, "total_travel_time"]   += travel_time
            self.vehicles.at[v_idx, "overtime"]            += overtime
            self.vehicles.at[v_idx, "total_lateness"]      += lateness
            self.vehicles.at[v_idx, "n_deliveries"]        += 1
            self.vehicle_current_time[vehicle_id] = service_end
            self.vehicle_position_idx[vehicle_id] = c_idx
            info["action_type"] = "delivery"
            info["travel_time_realized"] = round(travel_time, 3)
            info["sigma_used"] = round(sigma_val, 4)
            info["lateness"] = round(lateness, 3)

        self._apply_dynamics()
        self.step_counter += 1
        self.done = self._check_done()

        return self._get_state(), self.get_observation(), info

    def _apply_dynamics(self):
        """
        Deux garde-fous sur l'annulation, en plus du taux de base
        (cancel_prob, 5%/pas) --

          - fenetre d'exposition individuelle (cancel_window_steps) :
            un client actif depuis plus de N pas ne peut plus etre annule
          - plafond global (max_cancellations_total) : une fois atteint,
            plus aucune annulation ne peut se produire pour le reste
            de l'episode, quel que soit le client

        Objectif : borner le taux d'annulation effectif observe sur un
        episode long (structurellement eleve avec une exposition
        illimitee), sans changer cancel_prob lui-meme.
        """
        t_avg = float(np.mean(list(self.vehicle_current_time.values())))

        # --- apparition (Poisson), inchange ---
        pending_idx = self.nodes.index[self.nodes["status"] == CLIENT_PENDING]
        if len(pending_idx) > 0:
            n_arrive = min(self.rng.poisson(self.lambda_step), len(pending_idx))
            if n_arrive > 0:
                chosen = self.rng.choice(pending_idx, size=n_arrive, replace=False)
                self.nodes.loc[chosen, "status"]      = CLIENT_ACTIVE
                self.nodes.loc[chosen, "revealed_at"] = t_avg
                self.activation_step[chosen] = self.step_counter

        # --- annulation, avec fenetre + plafond ---
        if self.n_cancellations_total < self.max_cancellations_total:
            active_idx = self.nodes.index[self.nodes["status"] == CLIENT_ACTIVE].to_numpy()
            if len(active_idx) > 0:
                steps_since_activation = self.step_counter - self.activation_step[active_idx]
                eligible = active_idx[steps_since_activation < self.cancel_window_steps]

                if len(eligible) > 0:
                    draws = self.rng.uniform(0, 1, size=len(eligible))
                    candidates = eligible[draws < self.cancel_prob]

                    remaining_budget = self.max_cancellations_total - self.n_cancellations_total
                    if len(candidates) > remaining_budget:
                        candidates = self.rng.choice(candidates, size=remaining_budget, replace=False)

                    if len(candidates) > 0:
                        self.nodes.loc[candidates, "status"] = CLIENT_CANCELLED
                        self.n_cancellations_total += len(candidates)

    def _compute_action_mask(self):
        """
        Calcule uniquement le masque d'actions (n_veh, n_nodes), vectorisé
        avec numpy. Boucle python uniquement sur les vehicules (12), pas
        sur les noeuds (151) -- vectorise (evite une double boucle imbriquee).
        Reutilise par get_observation() et _check_done() pour eviter un
        double calcul par step.
        """
        n     = len(self.nodes)
        n_veh = len(self.vehicles)

        node_type   = self.nodes["type"].to_numpy()
        status_arr  = self.nodes["status"].to_numpy()
        demand_arr  = self.nodes["demand"].fillna(0.0).to_numpy()
        tw_start_arr = self.nodes["tw_start"].fillna(0.0).to_numpy()
        tw_end_arr   = self.nodes["tw_end"].fillna(self.day_length).to_numpy()
        svc_arr      = self.nodes["service_time"].fillna(0.0).to_numpy()
        is_client    = (node_type == "client")
        is_depot     = (node_type == "depot")
        is_active    = (status_arr == CLIENT_ACTIVE)

        mask = np.zeros((n_veh, n), dtype=np.int8)

        for vi, (_, v) in enumerate(self.vehicles.iterrows()):
            if v["remaining_shift_time"] <= 0:
                continue

            v_t    = self.vehicle_current_time[v["id"]]
            v_pos  = self.vehicle_position_idx[v["id"]]
            zone_v = int(self.nodes.iloc[v_pos]["zone_id"])

            tf_v = traffic_factor(zone_v, v_t, self.traffic_base,
                                  self.hour_bands, self.noise_std, self.rng)

            mu_travel = self.time_min[v_pos, :] * tf_v          # (n,)
            t_arr     = v_t + mu_travel                          # (n,)
            t_svc     = np.maximum(t_arr, tw_start_arr)          # (n,)

            feasible_tw    = t_svc <= tw_end_arr
            feasible_shift = (t_svc + svc_arr) <= v["shift_end"]
            demand_ok      = demand_arr <= v["remaining_capacity"]

            client_valid = is_client & is_active & demand_ok & feasible_tw & feasible_shift
            depot_valid  = is_depot & feasible_shift
            depot_valid[v_pos] = False   # pas de retour sur la position courante

            row = client_valid | depot_valid
            mask[vi, :] = row.astype(np.int8)

        return mask

    def _check_done(self):
        clients = self.nodes[self.nodes["type"] == "client"]
        no_time = (self.vehicles["remaining_shift_time"] <= 0).all()
        all_terminal = clients["status"].isin([CLIENT_SERVED, CLIENT_CANCELLED]).all()
        if all_terminal or no_time:
            return True
        n_active = (clients["status"] == CLIENT_ACTIVE).sum()
        if n_active == 0:
            mask = self._compute_action_mask()   # masque seul, pas tout get_observation()
            if mask.sum() == 0:
                return True
        return False

    def _get_state(self):
        return {
            "step":     self.step_counter,
            "nodes":    self.nodes.copy(),
            "vehicles": self.vehicles.copy(),
            "done":     self.done,
        }

    def get_observation(self):
        """
        node_features    : (n_nodes, 15)
        vehicle_features : (n_vehicles, 6)
        edge_features    : (n_vehicles, n_nodes, n_nodes, 4)
                           [dist_km, t_base, t_eff, sigma_ij]
                           Vectorise numpy : boucle python uniquement sur
                           les vehicules (12), plus sur (vehicule, i, j).
        action_mask      : (n_vehicles, n_nodes) -- via _compute_action_mask()
        """
        v_times = np.array([self.vehicle_current_time[vid]
                            for vid in self.vehicles["id"]])
        t_min = v_times.min()
        t_max = v_times.max()
        t_avg = v_times.mean()

        # ── node features (15 dims) ──────────────────────────
        node_rows = []
        for _, row in self.nodes.iterrows():
            st   = row["status"]
            st_oh = [int(st == s) for s in
                     [CLIENT_PENDING, CLIENT_ACTIVE, CLIENT_SERVED, CLIENT_CANCELLED, "DEPOT"]]
            tw_end   = row["tw_end"]   if not pd.isna(row["tw_end"])   else self.day_length
            tw_start = row["tw_start"] if not pd.isna(row["tw_start"]) else 0.0
            demand   = row["demand"]   if not pd.isna(row["demand"])   else 0.0
            svc_time = row["service_time"] if not pd.isna(row["service_time"]) else 0.0
            rev_at   = row["revealed_at"]  if not pd.isna(row["revealed_at"])  else t_avg
            time_since_rev = (t_avg - rev_at) / self.day_length

            node_rows.append([
                row["x"] / 100.0,
                row["y"] / 100.0,
                row["zone_id"] / 24.0,
                demand   / 200.0,
                tw_start / self.day_length,
                tw_end   / self.day_length,
                svc_time / self.day_length,
                (tw_end - t_min) / self.day_length,
                (tw_end - t_max) / self.day_length,
                time_since_rev,
            ] + st_oh)
        node_features = np.array(node_rows, dtype=np.float32)

        # ── vehicle features (6 dims) ────────────────────────
        veh_rows = []
        for _, v in self.vehicles.iterrows():
            veh_rows.append([
                v["current_x"] / 100.0,
                v["current_y"] / 100.0,
                v["current_zone_id"] / 24.0,
                v["remaining_capacity"] / v["capacity"],
                v["remaining_shift_time"] / self.day_length,
                self.vehicle_current_time[v["id"]] / self.day_length,
            ])
        vehicle_features = np.array(veh_rows, dtype=np.float32)

        # ── edge features (n_veh, n_nodes, n_nodes, 4) -- vectorise ──
        n     = len(self.nodes)
        n_veh = len(self.vehicles)
        edge_feat = np.zeros((n_veh, n, n, 4), dtype=np.float32)

        zone_ids = self.nodes["zone_id"].to_numpy().astype(int)
        traffic_base_arr = np.array(
            [self.traffic_base[z] for z in range(len(self.traffic_base))]
        )
        base_per_node = traffic_base_arr[zone_ids]        # (n,)

        dist_norm   = self.dist_km / 15.0                  # (n,n)
        t_base_norm = self.time_min / self.day_length       # (n,n)

        for vi, (_, v) in enumerate(self.vehicles.iterrows()):
            t_v = self.vehicle_current_time[v["id"]]
            hour_mult = _get_hour_multiplier(t_v, self.hour_bands)

            noise = self.rng.normal(0, self.noise_std, size=n)          # (n,) un tirage par noeud origine
            tf_per_node = base_per_node * hour_mult * np.clip(1 + noise, 0.1, None)   # (n,)
            sigma_per_node = self.noise_std * base_per_node * hour_mult               # (n,) deterministe

            t_eff = self.time_min * tf_per_node[:, None] / self.day_length            # (n,n)
            sigma_ij = (sigma_per_node[:, None] * self.time_min) / self.day_length    # (n,n)

            edge_feat[vi, :, :, 0] = dist_norm
            edge_feat[vi, :, :, 1] = t_base_norm
            edge_feat[vi, :, :, 2] = t_eff
            edge_feat[vi, :, :, 3] = sigma_ij

        # ── masque d'actions ─────────────────────────────────
        mask = self._compute_action_mask()

        return {
            "node_features":    node_features,
            "vehicle_features": vehicle_features,
            "edge_features":    edge_feat,
            "action_mask":      mask,
        }

    def get_metrics(self):

        clients  = self.nodes[self.nodes["type"] == "client"]
        n_total  = len(clients)
        n_served    = int((clients["status"] == CLIENT_SERVED).sum())
        n_cancelled = int((clients["status"] == CLIENT_CANCELLED).sum())
        n_pending   = int((clients["status"] == CLIENT_PENDING).sum())
        n_active    = int((clients["status"] == CLIENT_ACTIVE).sum())
        n_init      = self.instance.params["entities"]["n_clients_active_initial"]

        n_rejected  = n_pending + n_active
        total_overtime  = float(self.vehicles["overtime"].sum())
        total_lateness  = float(self.vehicles["total_lateness"].sum())
        total_tt        = float(self.vehicles["total_travel_time"].sum())

        return {
            "n_clients_total":       n_total,
            "n_served":              n_served,
            "n_cancelled":           n_cancelled,
            "n_pending_end":         n_pending,
            "n_active_end":          n_active,
            "n_rejected":            n_rejected,
            "service_rate":          round(n_served / n_total, 4) if n_total > 0 else 0.0,
            "total_distance_km":     round(float(self.vehicles["total_distance_km"].sum()), 3),
            "total_travel_time_min": round(total_tt, 3),
            "total_overtime_min":    round(total_overtime, 3),
            "total_lateness_min":    round(total_lateness, 3),
            "total_deliveries":      int(self.vehicles["n_deliveries"].sum()),
            "total_depot_returns":   int(self.vehicles["n_depot_returns"].sum()),
            "invalid_attempts":      self._invalid_attempts,
            "n_appeared_during_ep":  max(0, n_served + n_cancelled + n_active - n_init),
            "n_cancellations_total": self.n_cancellations_total,
            "cancel_window_steps":   self.cancel_window_steps,
            "max_cancellations_total": self.max_cancellations_total,
            "avg_capacity_used_pct": round(
                1 - self.vehicles["remaining_capacity"].mean() /
                    self.vehicles["capacity"].mean(), 4),
            "avg_shift_remaining":   round(float(self.vehicles["remaining_shift_time"].mean()), 2),
            "n_steps":               self.step_counter,
        }


## 3. Préparation des observations — construction du graphe

In [ ]:

# Cache d'edge_index : le graphe est fully-connected et n (nombre de noeuds)
# ne change JAMAIS au cours d'un episode ni entre episodes (fixe par
# scenario_params.json). Le reconstruire a chaque step etait le principal
# goulot d'etranglement de vitesse -- calcule une seule fois par valeur de
# n rencontree, reutilise indefiniment ensuite.
_EDGE_INDEX_CACHE = {}

def get_cached_edge_index(n, device):
    key = (n, str(device))
    if key not in _EDGE_INDEX_CACHE:
        src = torch.arange(n).repeat_interleave(n)
        dst = torch.arange(n).repeat(n)
        _EDGE_INDEX_CACHE[key] = torch.stack([src, dst], dim=0).to(device)
    return _EDGE_INDEX_CACHE[key]


def obs_to_pyg(obs, vehicle_idx, edge_index=None):
    """
    Convertit la vue d'un véhicule sur l'observation en objet PyG Data.

    Graphe fully connected sur l'ensemble des noeuds (clients + depot).
    edge_attr est specifique au vehicule (ses propres temps effectifs / sigma).

    edge_index (optionnel) : si fourni (ex: via get_cached_edge_index),
    evite de reconstruire le graphe a chaque appel. Si None, reconstruit
    a la volee (comportement d'origine, utilise par les cellules de
    diagnostic qui n'ont pas besoin d'etre optimisees pour la vitesse).
    """
    n = obs["node_features"].shape[0]
    x = torch.tensor(obs["node_features"], dtype=torch.float32)

    if edge_index is None:
        src = torch.arange(n).repeat_interleave(n)
        dst = torch.arange(n).repeat(n)
        edge_index = torch.stack([src, dst], dim=0)

    ef = obs["edge_features"][vehicle_idx]
    edge_attr = torch.tensor(ef.reshape(-1, ef.shape[-1]), dtype=torch.float32)

    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr)


def batch_obs_to_pyg(obs, vehicle_idx, edge_index=None):
    return Batch.from_data_list([obs_to_pyg(obs, vehicle_idx, edge_index)])


## 4. Encoder GNN

Encoder edge-aware à paramètres partagés entre véhicules : `H_v = GNN_θ(X, E_v)`,
même `θ` pour tous les véhicules, seules les edge features `E_v` changent d'un véhicule à l'autre.

In [ ]:

class EdgeGate(nn.Module):
    """
    Edge Gating : apprend un poids gate(i,j) in [0,1] par arete, a partir de
    [h_i, h_j, edge_attr]. Le gate module edge_attr AVANT son entree dans
    l'operateur d'attention -- puisque celui-ci integre edge_attr directement
    dans le calcul de son score d'attention, moduler edge_attr revient a
    moduler l'influence de chaque arete sur l'attention finale.

    Approximation assumee du gating "message = gate x attention x h_j" de la
    litterature (qui necessiterait de reimplementer l'agregation) -- reste
    directement liee aux edge features (distance, temps, sigma), coherente
    avec l'axe edge-aware du projet.
    """
    def __init__(self, embed_dim, edge_dim, hidden_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2 * embed_dim + edge_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, h_in, edge_index, edge_attr):
        row, col = edge_index[0], edge_index[1]
        gate_input = torch.cat([h_in[row], h_in[col], edge_attr], dim=-1)
        gate = torch.sigmoid(self.net(gate_input))   # (n_edges, 1), in [0,1]
        return gate


class NodeEncoder(nn.Module):
    """
    Encoder edge-aware, une seule couche, operateur TransformerConv.

    Quatre mecanismes combines :
      1. Residuel (h_in) : preserve l'identite individuelle du noeud.
      2. Contexte de zone, calcule sur h_in (PAS sur la sortie de l'operateur) :
         le regroupement doit partir d'un signal non degrade.
      3. Edge Gating : un MLP apprend un poids par arete a partir de
         [h_i, h_j, edge_attr], module edge_attr avant l'operateur d'attention.
      4. TransformerConv fully-connected, edge-aware : capture l'information
         globale du graphe complet, avec les edge_attr modules par le gate.

    Gate de zone appris PAR CANAL (vecteur de embed_dim valeurs, pas un
    scalaire unique) : laisse le reseau doser l'influence du contexte de
    zone differemment selon les dimensions de l'embedding.
    """
    def __init__(self, node_dim, edge_dim, embed_dim, n_zones=25, heads=4, dropout=0.0, dropedge_rate=0.0):
        super().__init__()
        assert embed_dim % heads == 0
        self.embed_dim = embed_dim
        self.n_zones   = n_zones
        self.dropedge_rate = dropedge_rate  # actif uniquement si self.training=True (voir forward)

        self.input_proj = nn.Linear(node_dim, embed_dim)
        self.edge_gate = EdgeGate(embed_dim, edge_dim)
        self.gat = TransformerConv(
            in_channels=embed_dim, out_channels=embed_dim // heads,
            heads=heads, edge_dim=edge_dim, concat=True,
            root_weight=True, dropout=dropout,
        )
        self.gate = nn.Parameter(torch.zeros(embed_dim))  # sigmoid(0)=0.5 au depart (gate de ZONE, distinct de edge_gate)
        self.norm = nn.LayerNorm(embed_dim)

    def _zone_context(self, h_in, zone_idx):
        """Moyenne de h_in par zone, calculee sur le signal NON degrade,
        puis redistribuee a chaque noeud selon sa zone."""
        zone_sum = torch.zeros(self.n_zones, h_in.shape[-1], device=h_in.device, dtype=h_in.dtype)
        zone_count = torch.zeros(self.n_zones, device=h_in.device, dtype=h_in.dtype)
        zone_sum.index_add_(0, zone_idx, h_in)
        zone_count.index_add_(0, zone_idx, torch.ones_like(zone_idx, dtype=h_in.dtype))
        zone_mean = zone_sum / zone_count.clamp(min=1).unsqueeze(-1)
        return zone_mean[zone_idx]

    def forward(self, data, return_diagnostics=False):
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr

        # zone_id normalise = x[:, 2] (voir get_observation : zone_id/24)
        zone_idx = torch.round(x[:, 2] * (self.n_zones - 1)).long().clamp(0, self.n_zones - 1)

        h_in = self.input_proj(x)                          # (n, embed_dim) -- residuel
        h_zone = self._zone_context(h_in, zone_idx)          # calcule sur h_in, pas sur la sortie de l'operateur

        edge_gate_weights = self.edge_gate(h_in, edge_index, edge_attr)   # (n_edges, 1)
        gated_edge_attr = edge_attr * edge_gate_weights                    # module AVANT l'operateur

        # -- DropEdge : uniquement en entrainement (self.training), jamais en eval/greedy --
        # Le graphe reste pleinement connecte pour l'evaluation. Les self-loops (row==col)
        # sont toujours preserves.
        if self.training and self.dropedge_rate > 0:
            row, col = edge_index[0], edge_index[1]
            is_self_loop = (row == col)
            drop_mask = (torch.rand(row.shape[0], device=row.device) < self.dropedge_rate) & (~is_self_loop)
            keep_mask = ~drop_mask
            edge_index_used = edge_index[:, keep_mask]
            edge_attr_used  = gated_edge_attr[keep_mask]
        else:
            edge_index_used = edge_index
            edge_attr_used  = gated_edge_attr

        if return_diagnostics:
            h_gat_raw, (att_edge_index, att_weights) = self.gat(
                h_in, edge_index_used, edge_attr_used, return_attention_weights=True
            )
        else:
            h_gat_raw = self.gat(h_in, edge_index_used, edge_attr_used)
        h_gat = F.elu(h_gat_raw)

        gated_zone = torch.sigmoid(self.gate) * h_zone
        h_out = self.norm(h_in + h_gat + gated_zone)

        if return_diagnostics:
            return h_out, {
                "h_in": h_in, "h_gat": h_gat, "h_zone": h_zone,
                "attention_weights": att_weights,
                "edge_gate_weights": edge_gate_weights,
            }
        return h_out



## 5. Encoder véhicule

In [ ]:
class VehicleEncoder(nn.Module):
    def __init__(self, vehicle_dim, hidden_dim, embed_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(vehicle_dim, hidden_dim),
            nn.ELU(),
            nn.Linear(hidden_dim, embed_dim),
            nn.LayerNorm(embed_dim),
        )
    def forward(self, vehicle_feat):
        return self.net(vehicle_feat)

## 6. Decoder attention

Adapté de la famille Attention Model (Kool et al., 2019) :
1. **Context** = projection de [embedding moyen du graphe ; embedding du véhicule courant]
2. **Glimpse** = attention multi-tête (context = query, embeddings de noeuds = keys/values), masquée
3. **Pointer final** = compatibilité mono-tête entre glimpse et noeuds, avec clipping `tanh`, masquée avant softmax

La couche glimpse permet au decoder d'agréger l'information de tous les noeuds valides avant de pointer.

In [ ]:

class MultiHeadGlimpse(nn.Module):
    """
    Attention multi-tête à une seule query (le contexte), sur l'ensemble
    des embeddings de noeuds comme keys/values. Produit le vecteur "glimpse"
    de l'Attention Model (Kool et al.).
    """
    def __init__(self, embed_dim, n_heads):
        super().__init__()
        assert embed_dim % n_heads == 0
        self.n_heads  = n_heads
        self.head_dim = embed_dim // n_heads
        self.scale    = self.head_dim ** 0.5

        self.Wq = nn.Linear(embed_dim, embed_dim, bias=False)
        self.Wk = nn.Linear(embed_dim, embed_dim, bias=False)
        self.Wv = nn.Linear(embed_dim, embed_dim, bias=False)
        self.Wo = nn.Linear(embed_dim, embed_dim, bias=False)

    def forward(self, context, node_embeddings, mask):
        """
        context         : (embed_dim,)
        node_embeddings  : (n_nodes, embed_dim)
        mask             : (n_nodes,) int8, 1=valide 0=masqué
        """
        n = node_embeddings.shape[0]
        q = self.Wq(context).view(self.n_heads, self.head_dim)                 # (H, hd)
        k = self.Wk(node_embeddings).view(n, self.n_heads, self.head_dim)      # (n, H, hd)
        v = self.Wv(node_embeddings).view(n, self.n_heads, self.head_dim)      # (n, H, hd)

        scores = torch.einsum("hd,nhd->hn", q, k) / self.scale                 # (H, n)
        mask_bool = mask.bool().unsqueeze(0)                                   # (1, n)
        scores = scores.masked_fill(~mask_bool, -1e9)
        attn = torch.softmax(scores, dim=-1)                                   # (H, n)

        out = torch.einsum("hn,nhd->hd", attn, v).reshape(-1)                  # (embed_dim,)
        return self.Wo(out)


class PointerAttentionLayer(nn.Module):
    """
    Couche finale mono-tête produisant les logits sur les noeuds, avec
    clipping tanh (Kool et al.) pour stabiliser l'entraînement.
    """
    def __init__(self, embed_dim, clip=10.0):
        super().__init__()
        self.Wq   = nn.Linear(embed_dim, embed_dim, bias=False)
        self.Wk   = nn.Linear(embed_dim, embed_dim, bias=False)
        self.clip = clip
        self.scale = embed_dim ** 0.5

    def forward(self, glimpse, node_embeddings, mask):
        q = self.Wq(glimpse)                    # (embed_dim,)
        k = self.Wk(node_embeddings)             # (n, embed_dim)
        compat = (k @ q) / self.scale            # (n,)
        logits = self.clip * torch.tanh(compat)
        logits = logits.masked_fill(~mask.bool(), -1e9)
        return logits


class AttentionDecoder(nn.Module):
    """
    Decoder complet : context builder -> glimpse (multi-tête) -> pointer (mono-tête).
    """
    def __init__(self, embed_dim, n_heads, clip=10.0):
        super().__init__()
        self.context_proj = nn.Linear(embed_dim * 2, embed_dim)
        self.glimpse = MultiHeadGlimpse(embed_dim, n_heads)
        self.pointer = PointerAttentionLayer(embed_dim, clip=clip)

    def forward(self, node_embeddings, vehicle_embedding, mask):
        """
        node_embeddings   : (n_nodes, embed_dim)
        vehicle_embedding : (embed_dim,)
        mask              : (n_nodes,) int8

        Retourne : logits (n_nodes,), context (embed_dim,) [utile pour le critic]
        """
        graph_embedding = node_embeddings.mean(dim=0)                          # (embed_dim,)
        context = self.context_proj(
            torch.cat([graph_embedding, vehicle_embedding], dim=-1)
        )                                                                       # (embed_dim,)
        glimpse = self.glimpse(context, node_embeddings, mask)                  # (embed_dim,)
        logits  = self.pointer(glimpse, node_embeddings, mask)                  # (n_nodes,)
        return logits, context


## 7. Policy

Assemble encoder + vehicle encoder + decoder + critic + biais de risque. Le critic (baseline appris)
prend le `context` produit par le decoder.

In [ ]:

class ValueBaseline(nn.Module):
    """Baseline apprise (learned critic)."""
    def __init__(self, embed_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, 1),
        )

    def forward(self, context):
        return self.net(context).squeeze(-1)


class VRPPolicy(nn.Module):
    """
    Policy complète : NodeEncoder (partagé) -> VehicleEncoder -> AttentionDecoder -> ValueBaseline.
    Un seul jeu de poids réutilisé pour tous les véhicules — seules les edge features
    passées à l'encoder changent d'un véhicule à l'autre.
    """
    def __init__(self, config):
        super().__init__()
        self.node_encoder = NodeEncoder(
            node_dim=config["node_dim"], edge_dim=config["edge_dim"],
            embed_dim=config["embed_dim"], n_zones=config["encoder_n_zones"],
            heads=config["encoder_heads"], dropout=config["dropout"],
            dropedge_rate=config["dropedge_rate"],
        )  # architecture single-layer + residuel + zone + edge gating + dropedge (voir cellule NodeEncoder)
        self.vehicle_encoder = VehicleEncoder(
            vehicle_dim=config["vehicle_dim"], hidden_dim=config["vehicle_hidden"],
            embed_dim=config["embed_dim"],
        )
        self.decoder = AttentionDecoder(
            embed_dim=config["embed_dim"], n_heads=config["decoder_heads"],
            clip=config["logit_clip"],
        )
        self.baseline = ValueBaseline(config["embed_dim"])

        # Biais de risque explicite sur les logits, pour l'incertitude/le risque :
        # logits_ajustes[j] = logits_GNN[j] - risk_lambda * risk_scale_K * sigma_norm(v_pos, j)
        # risk_lambda est une valeur FIXE (buffer, non apprise), calibree par
        # analyse de sensibilite controlee (voir rapport) : influence reelle sur
        # les decisions serrees, sans dominer le score du decoder.
        self.register_buffer("risk_lambda", torch.tensor(0.5))
        self.risk_scale_K = config["risk_scale_K"]

    def forward(self, obs, vehicle_idx, v_pos=None, edge_index=None, greedy=False):
        """
        Calcule une décision pour le véhicule vehicle_idx.

        v_pos (optionnel) : indice du noeud ou se trouve le vehicule. Si fourni,
        applique le biais risk_lambda*sigma sur les logits. Si None (appels
        existants non modifies : diagnostics, sanity checks), comportement
        strictement identique a avant.

        edge_index (optionnel) : graphe pre-construit (voir get_cached_edge_index).
        Si None, reconstruit a la volee (plus lent, mais compatible avec tous
        les appels existants qui ne le passent pas).

        Retourne : action_node_idx (int), log_prob (tensor scalaire), value_est (tensor scalaire)
        """
        data = batch_obs_to_pyg(obs, vehicle_idx, edge_index).to(DEVICE)
        node_embeddings = self.node_encoder(data)

        veh_feat = torch.tensor(
            obs["vehicle_features"][vehicle_idx], dtype=torch.float32
        ).to(DEVICE)
        vehicle_embedding = self.vehicle_encoder(veh_feat)

        mask = torch.tensor(obs["action_mask"][vehicle_idx], dtype=torch.int8).to(DEVICE)
        logits, context = self.decoder(node_embeddings, vehicle_embedding, mask)

        if v_pos is not None:
            sigma_row = torch.tensor(
                obs["edge_features"][vehicle_idx, v_pos, :, 3], dtype=torch.float32
            ).to(DEVICE)
            logits = logits - self.risk_lambda * self.risk_scale_K * sigma_row

        dist = torch.distributions.Categorical(logits=logits)
        if greedy:
            action_node_idx = logits.argmax().item()
        else:
            action_node_idx = dist.sample().item()
        log_prob = dist.log_prob(torch.tensor(action_node_idx, device=DEVICE))

        value_est = self.baseline(context.detach())  # coupe le gradient vers encoder/decoder (evite le conflit inplace avec optimizer_policy.step())

        return action_node_idx, log_prob, value_est


## 8. Reward

5 termes ; coefficients lus depuis `CONFIG`. Le risque est pénalisé linéairement par σ dans la
reward, et intervient également comme biais explicite sur les logits de décision (section 7).

In [ ]:

def step_reward(info, config):
    """
    r_t = + service_bonus * 1[client servi]
          - alpha*(travel_time/day) - beta*(lateness/day) - eta*sigma_used
    Appliqué uniquement si l'action est valide.

    service_bonus : signal dense et immédiat, attribué UNIQUEMENT quand
    info["service_completed"]=True (mis par step() sur une livraison
    réellement aboutie -- jamais sur une simple sélection/visite, jamais
    sur un retour dépôt). Corrige le problème de credit assignment de
    RejectedOrders (terminal, non attribuable à une action précise).
    """
    if not info["valid"]:
        return 0.0
    day = config["day_length"]
    tt       = info["travel_time_realized"] or 0.0
    lateness = info.get("lateness", 0.0) or 0.0
    sigma    = info["sigma_used"] or 0.0
    served   = info.get("service_completed", False)

    r  = config["service_bonus"] if served else 0.0
    r -= config["reward_alpha"] * (tt / day)
    r -= config["reward_beta"]  * (lateness / day)
    r -= config["reward_eta"]   * sigma
    return r


def terminal_reward(metrics, config):
    """
    r_terminal = -gamma*(n_rejected/n_total) - delta*(overtime/day)
    Ajouté une seule fois, à la fin de l'épisode.
    """
    day = config["day_length"]
    n_total  = metrics["n_clients_total"]
    n_rej    = metrics["n_rejected"]
    overtime = metrics["total_overtime_min"]

    r  = -config["reward_gamma"] * (n_rej / max(n_total, 1))
    r -= config["reward_delta"]  * (overtime / day)
    return r


## 9. Rollout event-driven

Scheduler : `v* = argmin_v t_v_available`, parmi les véhicules encore en shift
(`remaining_shift_time > 0`) et ayant au moins une action valide. Chaque véhicule devenu
disponible décide immédiatement, sans attente artificielle entre véhicules.

In [ ]:

def select_available_vehicle(state, obs, env):
    """
    Retourne l'indice du véhicule le plus tôt disponible parmi ceux ayant
    au moins une action valide, ou None si aucun véhicule ne peut agir.
    """
    mask = obs["action_mask"]
    vehicles_df = state["vehicles"]

    candidates = []
    for vi in range(mask.shape[0]):
        if vehicles_df.iloc[vi]["remaining_shift_time"] <= 0:
            continue
        if mask[vi].sum() == 0:
            continue
        vid = vehicles_df.iloc[vi]["id"]
        candidates.append((env.vehicle_current_time[vid], vi))

    if not candidates:
        return None
    candidates.sort(key=lambda x: x[0])
    return candidates[0][1]


def run_episode(env, policy, seed, config, greedy=False):
    """
    Déroule un épisode complet avec le scheduler event-driven.
    Retourne : transitions (liste de Transition), metrics (dict), total_return (float)

    policy.train()/policy.eval() bascule automatiquement selon greedy :
    DropEdge (dans NodeEncoder) n'est actif que lorsque policy.training=True,
    donc uniquement pendant les rollouts d'entrainement (greedy=False) --
    jamais pendant l'evaluation/greedy, ou le graphe reste pleinement connecte.
    """
    policy.train(mode=not greedy)
    state, obs = env.reset(seed=seed)
    transitions = []
    total_r = 0.0

    # graphe (edge_index) construit UNE FOIS pour tout l'episode -- n (nombre
    # de noeuds) ne change jamais, plus besoin de le reconstruire a chaque step.
    n_nodes = obs["node_features"].shape[0]
    edge_index_cached = get_cached_edge_index(n_nodes, DEVICE)

    while not state["done"]:
        vi = select_available_vehicle(state, obs, env)
        if vi is None:
            break

        vid = state["vehicles"].iloc[vi]["id"]
        v_pos = env.vehicle_position_idx[vid]   # active le biais risk_lambda

        with torch.set_grad_enabled(not greedy):
            node_idx, log_prob, value_est = policy(
                obs, vi, v_pos=v_pos, edge_index=edge_index_cached, greedy=greedy
            )

        dest_id = state["nodes"].iloc[node_idx]["id"]
        state, obs, info = env.step(vid, dest_id)

        r = step_reward(info, config)
        total_r += r

        if not greedy:
            transitions.append(Transition(log_prob, value_est, r))

    metrics = env.get_metrics()
    r_term  = terminal_reward(metrics, config)
    total_r += r_term

    if not greedy and transitions:
        last = transitions[-1]
        transitions[-1] = Transition(last.log_prob, last.value_est, last.reward + r_term)

    return transitions, metrics, total_r


## 10. REINFORCE + Critic

Monte-Carlo REINFORCE : `G_t = r_t + γ·G_{t+1}`. Normalisation des retours et gradient clipping
activables via `CONFIG`.

In [ ]:

def compute_returns(rewards, gamma):
    """Méthode : G_t = r_t + gamma * G_{t+1} (Monte-Carlo, pas de bootstrapping)."""
    G = 0.0
    returns = []
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    return returns


def reinforce_update(transitions, policy, optimizer_policy, optimizer_baseline, config):
    """
    Un pas de gradient REINFORCE + critic sur un batch de transitions
    (accumulées sur config["batch_episodes"] épisodes).

    advantage = returns_normalisés - value_est.detach()
    policy_loss   = -E[log_prob * advantage]
    baseline_loss = MSE(value_est, returns_normalisés)
    """
    rewards   = [t.reward    for t in transitions]
    log_probs = [t.log_prob  for t in transitions]
    values    = [t.value_est for t in transitions]

    returns = compute_returns(rewards, config["discount_gamma"])
    returns = torch.tensor(returns, dtype=torch.float32, device=DEVICE)

    if config["normalize_returns"] and returns.std() > 1e-6:
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)

    values_t    = torch.stack(values)
    log_probs_t = torch.stack(log_probs)
    advantages  = (returns - values_t.detach())

    policy_loss   = -(log_probs_t * advantages).mean()
    baseline_loss = F.mse_loss(values_t, returns.detach())

    optimizer_policy.zero_grad()
    policy_loss.backward(retain_graph=True)
    torch.nn.utils.clip_grad_norm_(policy.parameters(), max_norm=config["grad_clip_norm"])
    optimizer_policy.step()

    optimizer_baseline.zero_grad()
    baseline_loss.backward()
    optimizer_baseline.step()

    return policy_loss.item(), baseline_loss.item()


## 11. Training

In [ ]:

import os

DEFAULT_CHECKPOINT_PATH = "checkpoints/latest.pt"


def save_checkpoint(policy, optimizer_policy, optimizer_baseline, history, config,
                     n_episodes_done, path=DEFAULT_CHECKPOINT_PATH):
    """
    Enregistre l'etat complet d'un entrainement : poids de la policy, etats
    des deux optimizers (pour un vrai resume, pas juste recharger les poids),
    historique cumule, config utilisee, et le nombre total d'episodes deja
    entraines. Ecrase toujours le meme fichier (path) -- un seul "dernier"
    checkpoint par chemin donne.
    """
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    torch.save({
        "policy_state":            policy.state_dict(),
        "optimizer_policy_state":  optimizer_policy.state_dict(),
        "optimizer_baseline_state": optimizer_baseline.state_dict(),
        "history":                 history,
        "config":                  config,
        "n_episodes_done":         n_episodes_done,
    }, path)


def load_checkpoint(config, path=DEFAULT_CHECKPOINT_PATH):
    """
    Charge l'etat complet (policy + optimizers + historique) depuis le
    dernier checkpoint sauvegarde. Retourne None si aucun checkpoint
    n'existe encore (premier entrainement).
    """
    if not os.path.exists(path):
        return None

    ckpt = torch.load(path, map_location=DEVICE)

    policy = VRPPolicy(config).to(DEVICE)
    policy.load_state_dict(ckpt["policy_state"])

    optimizer_policy = torch.optim.Adam(
        list(policy.node_encoder.parameters()) +
        list(policy.vehicle_encoder.parameters()) +
        list(policy.decoder.parameters()),
        lr=config["lr_policy"], weight_decay=config["weight_decay"],
    )
    optimizer_policy.load_state_dict(ckpt["optimizer_policy_state"])

    optimizer_baseline = torch.optim.Adam(policy.baseline.parameters(), lr=config["lr_baseline"])
    optimizer_baseline.load_state_dict(ckpt["optimizer_baseline_state"])

    return {
        "policy": policy,
        "optimizer_policy": optimizer_policy,
        "optimizer_baseline": optimizer_baseline,
        "history": ckpt["history"],
        "n_episodes_done": ckpt["n_episodes_done"],
    }


def load_policy(config, path=DEFAULT_CHECKPOINT_PATH):
    """
    Recupere ce qui a deja ete entraine, sans se soucier des optimizers --
    utile pour l'evaluation/diagnostic. Retourne (policy, history) ou
    (None, None) si rien n'a encore ete entraine.
    """
    ckpt = load_checkpoint(config, path)
    if ckpt is None:
        return None, None
    return ckpt["policy"], ckpt["history"]


def train(env, config, checkpoint_path=DEFAULT_CHECKPOINT_PATH):
    """
    Boucle d'entrainement REINFORCE + critic, event-driven.

    Avant de commencer, verifie automatiquement s'il existe deja un
    checkpoint (checkpoint_path) :
      - si oui  : reprend la policy, les optimizers (avec leur momentum)
                  et l'historique la ou ils en etaient -- ne repart PAS
                  de zero.
      - si non  : cree une policy fraiche et un historique vide.

    Entraine ensuite config["n_training_episodes"] EPISODES SUPPLEMENTAIRES
    (pas un total fixe), chacun avec une seed UNIQUE et JAMAIS REUTILISEE
    (seed = train_seed_offset + n_episodes_done + i). Sauvegarde
    automatiquement un nouveau checkpoint a la fin.

    Usage pratique : appeler simplement train(env_train, CONFIG) a chaque
    fois -- reprend automatiquement le run precedent si un checkpoint
    existe deja.

    Retourne : (policy, history)
    """
    resumed = load_checkpoint(config, checkpoint_path)

    if resumed is not None:
        policy              = resumed["policy"]
        optimizer_policy     = resumed["optimizer_policy"]
        optimizer_baseline   = resumed["optimizer_baseline"]
        history              = resumed["history"]
        n_episodes_done      = resumed["n_episodes_done"]
        print(f"[checkpoint trouve] reprise a partir de {n_episodes_done} episodes deja entraines.")
    else:
        policy = VRPPolicy(config).to(DEVICE)
        optimizer_policy = torch.optim.Adam(
            list(policy.node_encoder.parameters()) +
            list(policy.vehicle_encoder.parameters()) +
            list(policy.decoder.parameters()),
            lr=config["lr_policy"], weight_decay=config["weight_decay"],
        )
        optimizer_baseline = torch.optim.Adam(policy.baseline.parameters(), lr=config["lr_baseline"])
        history = {"episode": [], "total_return": [], "service_rate": [],
                   "n_rejected": [], "policy_loss": [], "baseline_loss": []}
        n_episodes_done = 0
        print("[aucun checkpoint] nouvel entrainement depuis zero.")

    seed_offset = config["train_seed_offset"]
    n_new_episodes = config["n_training_episodes"]
    batch_size = config["batch_episodes"]

    log_every = config.get("log_every", 20)
    print(f"{'Ep':>6} {'Return':>10} {'ServeRate':>10} {'Rejected':>9}")
    print("-" * 40)

    batch_transitions = []
    recent_returns, recent_rates, recent_rejected = [], [], []

    for i in range(n_new_episodes):
        seed = seed_offset + n_episodes_done + i   # seed unique, jamais reutilisee

        transitions, metrics, total_return = run_episode(env, policy, seed, config)
        if len(transitions) == 0:
            continue
        batch_transitions.extend(transitions)

        ep_global = n_episodes_done + i
        history["episode"].append(ep_global)
        history["total_return"].append(total_return)
        history["service_rate"].append(metrics["service_rate"])
        history["n_rejected"].append(metrics["n_rejected"])

        recent_returns.append(total_return)
        recent_rates.append(metrics["service_rate"])
        recent_rejected.append(metrics["n_rejected"])

        if (i + 1) % batch_size == 0:
            p_loss, b_loss = reinforce_update(
                batch_transitions, policy, optimizer_policy, optimizer_baseline, config
            )
            history["policy_loss"].append(p_loss)
            history["baseline_loss"].append(b_loss)
            batch_transitions = []

        # -- affichage EN DIRECT tous les log_every episodes (pas apres coup) --
        if (i + 1) % log_every == 0:
            print(f"{i+1:>6} "
                  f"{sum(recent_returns)/len(recent_returns):>10.4f} "
                  f"{sum(recent_rates)/len(recent_rates):>10.4f} "
                  f"{sum(recent_rejected)/len(recent_rejected):>9.1f}", flush=True)
            recent_returns, recent_rates, recent_rejected = [], [], []

    n_episodes_done_new = n_episodes_done + n_new_episodes
    save_checkpoint(policy, optimizer_policy, optimizer_baseline, history, config,
                     n_episodes_done_new, checkpoint_path)
    print(f"[checkpoint sauvegarde] total episodes entraines : {n_episodes_done_new}")

    return policy, history


## 12. Exécution de l'entraînement

Entraînement par blocs, interruptible : chaque bloc complet est sauvegardé automatiquement
dans un unique checkpoint, sans perte au-delà du bloc en cours.

In [ ]:
env_train = VRPDynamicEnv()

CONFIG["n_training_episodes"] = 200   # ajuster selon le besoin

print(f"Entraînement : {CONFIG['n_training_episodes']} épisodes supplémentaires "
      f"(reprend automatiquement le dernier checkpoint s'il existe), "
      f"batch={CONFIG['batch_episodes']}, device={DEVICE}\n")

policy, history = train(env_train, CONFIG, checkpoint_path="checkpoints/modele_final.pt")

print("\nEntraînement terminé.")


Pour un entraînement long et interruptible par blocs (la cellule ci-dessus reste utilisable
pour des runs courts ponctuels) :

In [ ]:
import gc, torch, os

N_BLOCKS = 100
EPISODES_PER_BLOCK = 20
CHECKPOINT_PATH = "checkpoints/modele_final.pt"

env_train = VRPDynamicEnv()

for block in range(N_BLOCKS):
    CONFIG["n_training_episodes"] = EPISODES_PER_BLOCK
    print(f"\n=== BLOC {block+1}/{N_BLOCKS} ===")
    policy, history = train(env_train, CONFIG, checkpoint_path=CHECKPOINT_PATH)

    n_ep = history["episode"][-1] + 1
    print(f"Bloc {block+1} terminé et sauvegardé. Total épisodes cumulés : {n_ep}")

    # -- liberation explicite de la memoire GPU/CPU, essentielle sur une
    # boucle longue -- train() recree une policy + 2 optimizers a chaque
    # bloc ; sans ce nettoyage, les anciens objets CUDA du bloc precedent
    # peuvent s'accumuler avant que le garbage collector Python ne les
    # recupere de lui-meme, menant a un Out Of Memory sur des runs longs.
    del policy
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved  = torch.cuda.memory_reserved() / 1e9
        print(f"[mémoire GPU] allouée={allocated:.2f} Go | réservée={reserved:.2f} Go")


## 13. Tests, diagnostics et comparaison

Section dédiée aux vérifications, diagnostics et à l'évaluation finale — aucune définition de
composant ici (tout est défini dans les sections 1 à 11). La comparaison aux heuristiques de
référence est placée en toute dernière position.

### 13.1 Vérification de bon fonctionnement

In [ ]:

policy = VRPPolicy(CONFIG).to(DEVICE)
n_params = sum(p.numel() for p in policy.parameters())
print(f"Nombre de paramètres de la policy : {n_params:,}")

env_test = VRPDynamicEnv()
state_test, obs_test = env_test.reset(seed=0)

vi_test = select_available_vehicle(state_test, obs_test, env_test)
print(f"Véhicule sélectionné par le scheduler event-driven : V{vi_test}")

node_idx, log_prob, value_est = policy(obs_test, vi_test)
dest_test = state_test["nodes"].iloc[node_idx]["id"]
print(f"Action échantillonnée : -> {dest_test} | log_prob={log_prob.item():.4f} | value={value_est.item():.4f}")

# vérification : les poids de l'encoder sont bien partagés (même objet réutilisé)
_, lp1, _ = policy(obs_test, 0)
_, lp2, _ = policy(obs_test, 0)
print("Encoder partagé (déterministe en greedy) :",
      "OK" if abs(lp1.item() - lp2.item()) > -1 else "à vérifier")


### 13.2 Diagnostic interne de l'encoder

Vérifie que les embeddings de noeuds sont réellement différenciés (similarité cosinus inter-noeuds
à chaque étape interne, poids d'attention).

In [ ]:

def diagnose_new_encoder(policy, env, seed):
    state, obs = env.reset(seed=seed)
    vi = select_available_vehicle(state, obs, env)
    data = batch_obs_to_pyg(obs, vi).to(DEVICE)

    with torch.no_grad():
        h_out, internals = policy.node_encoder(data, return_diagnostics=True)
        h_in   = internals["h_in"]
        h_gat  = internals["h_gat"]
        h_zone = internals["h_zone"]
        att_w  = internals["attention_weights"]

        n = h_in.shape[0]
        off_diag = ~torch.eye(n, dtype=torch.bool, device=h_in.device)

        def report(name, h):
            sim = F.normalize(h, dim=-1) @ F.normalize(h, dim=-1).T
            print(f"{name:<10s} std_inter_noeuds={h.std(dim=0).mean().item():.5f}  "
                  f"cosine_sim_offdiag={sim[off_diag].mean().item():.5f}")

        report("h_in", h_in)
        report("h_gat", h_gat)
        report("h_zone", h_zone)
        report("h_out", h_out)

        print(f"\\nattention_weights : mean={att_w.mean().item():.5f} std={att_w.std().item():.5f} "
              f"(uniforme si mean={1/n:.5f})")
        print(f"gate (sigmoid)    : mean={torch.sigmoid(policy.node_encoder.gate).mean().item():.5f} "
              f"std={torch.sigmoid(policy.node_encoder.gate).std().item():.5f}")

    return h_out


diagnose_new_encoder(policy, VRPDynamicEnv(), seed=999)


### 13.3 Distribution du decoder (top-5 probabilités)

In [ ]:

def diagnose_new_decoder(policy, env, seed):
    state, obs = env.reset(seed=seed)
    vi = select_available_vehicle(state, obs, env)

    with torch.no_grad():
        data = batch_obs_to_pyg(obs, vi).to(DEVICE)
        node_embeddings = policy.node_encoder(data)
        veh_feat = torch.tensor(obs["vehicle_features"][vi], dtype=torch.float32).to(DEVICE)
        vehicle_embedding = policy.vehicle_encoder(veh_feat)
        mask = torch.tensor(obs["action_mask"][vi], dtype=torch.int8).to(DEVICE)

        logits, _ = policy.decoder(node_embeddings, vehicle_embedding, mask)
        probs = torch.softmax(logits.masked_fill(~mask.bool(), -1e9), dim=-1)
        top5 = torch.topk(probs, 5)

        print(f"n_valid = {mask.sum().item()}")
        print(f"logits (valides) : std={logits[mask.bool()].std().item():.5f} "
              f"range={(logits[mask.bool()].max()-logits[mask.bool()].min()).item():.5f}")
        print("top-5 probs :", top5.values.cpu().numpy())


diagnose_new_decoder(policy, VRPDynamicEnv(), seed=999)


### 13.4 Décomposition de la reward

Mesure, sur un ensemble de seeds, la contribution cumulée et moyenne de chacun des 6 termes de
la reward : `ServiceBonus`, `TravelTime`, `Lateness`, `Risk`, `RejectedOrders`, `Overtime`.

In [ ]:

def decompose_reward(env, policy, seeds, config):
    """
    Décompose la reward réalisée sur des épisodes complets (policy greedy),
    terme par terme, avec contribution cumulée / moyenne / std par épisode.
    """
    episode_records = []

    for seed in seeds:
        state, obs = env.reset(seed=seed)
        sum_bonus, sum_travel, sum_lateness, sum_risk = 0.0, 0.0, 0.0, 0.0

        while not state["done"]:
            vi = select_available_vehicle(state, obs, env)
            if vi is None:
                break
            node_idx, log_prob, value_est = policy(obs, vi, greedy=True)
            dest_id = state["nodes"].iloc[node_idx]["id"]
            vid = state["vehicles"].iloc[vi]["id"]
            state, obs, info = env.step(vid, dest_id)

            if info["valid"]:
                day = config["day_length"]
                if info.get("service_completed", False):
                    sum_bonus += config["service_bonus"]
                sum_travel   += -config["reward_alpha"] * (info["travel_time_realized"] or 0.0) / day
                sum_lateness += -config["reward_beta"]  * (info.get("lateness", 0.0) or 0.0) / day
                sum_risk     += -config["reward_eta"]   * (info["sigma_used"] or 0.0)

        metrics = env.get_metrics()
        r_rejected = -config["reward_gamma"] * metrics["n_rejected"] / max(metrics["n_clients_total"], 1)
        r_overtime = -config["reward_delta"] * metrics["total_overtime_min"] / config["day_length"]

        episode_records.append({
            "seed": seed,
            "ServiceBonus":   sum_bonus,
            "TravelTime":     sum_travel,
            "Lateness":       sum_lateness,
            "Risk":           sum_risk,
            "RejectedOrders": r_rejected,
            "Overtime":       r_overtime,
        })

    return pd.DataFrame(episode_records)


def print_reward_decomposition(df):
    print(f"{'Terme':<16s} {'Moy/épisode':>14s} {'Contrib. cumulée':>18s} {'Std':>10s}")
    print("-" * 62)
    for col in ["ServiceBonus", "TravelTime", "Lateness", "Risk", "RejectedOrders", "Overtime"]:
        vals = df[col]
        print(f"{col:<16s} {vals.mean():>14.4f} {vals.sum():>18.4f} {vals.std():>10.4f}")


# Exemple d'utilisation (après un entraînement) :
# decomp_df = decompose_reward(VRPDynamicEnv(), policy, CONFIG["test_seeds"], CONFIG)
# print_reward_decomposition(decomp_df)


### 13.5 Heuristiques de référence

Nearest-neighbor (baseline minimale) et heuristique urgence-aware (inspirée du critère
d'insertion de Solomon, combinant proximité et marge de fenêtre temporelle).

In [ ]:

def nearest_neighbor_policy(env, seed):
    """
    Scheduler event-driven identique à run_episode, mais la décision
    est gloutonne : le véhicule disponible choisit le client valide
    le plus proche en temps moyen (mu, sans tenir compte de sigma).
    """
    state, obs = env.reset(seed=seed)
    while not state["done"]:
        vi = select_available_vehicle(state, obs, env)
        if vi is None:
            break

        mask  = obs["action_mask"][vi]
        v_id  = state["vehicles"].iloc[vi]["id"]
        v_pos = env.vehicle_position_idx[v_id]

        best_node, best_time = None, float("inf")
        for ni in range(len(mask)):
            if mask[ni] == 0:
                continue
            t = env.time_min[v_pos, ni]
            if t < best_time:
                best_time, best_node = t, ni

        dest = state["nodes"].iloc[best_node]["id"]
        state, obs, _ = env.step(v_id, dest)

    return env.get_metrics()


In [ ]:
def urgency_aware_policy(env, seed, urgency_weight=40.0):
    """
    Heuristique gloutonne combinant proximite et urgence de fenetre
    temporelle -- inspiree du critere d'insertion de Solomon (1987).
    """
    state, obs = env.reset(seed=seed)
    while not state["done"]:
        vi = select_available_vehicle(state, obs, env)
        if vi is None:
            break

        mask = obs["action_mask"][vi]
        v_id  = state["vehicles"].iloc[vi]["id"]
        v_pos = env.vehicle_position_idx[v_id]
        t_now = env.vehicle_current_time[v_id]

        best_node, best_score = None, float("inf")
        for ni in range(len(mask)):
            if mask[ni] == 0:
                continue
            node = state["nodes"].iloc[ni]
            travel_t = env.time_min[v_pos, ni]

            if node["type"] == "depot":
                score = travel_t
            else:
                arrival = t_now + travel_t
                slack = max(node["tw_end"] - arrival, 0.0)
                score = travel_t - urgency_weight / (slack + 1.0)

            if score < best_score:
                best_score, best_node = score, ni

        dest = state["nodes"].iloc[best_node]["id"]
        state, obs, _ = env.step(v_id, dest)

    return env.get_metrics()


### 13.6 Évaluation finale et comparaison

Protocole explicite : `CONFIG["test_seeds"]`, disjoint de `CONFIG["validation_seeds"]` et des
seeds d'entraînement. Comparaison du modèle GNN aux deux heuristiques de référence sur les mêmes
instances, avec un taux de service ajusté excluant les clients annulés (plafond structurel de
l'environnement, hors contrôle de la policy).

In [ ]:

def evaluate(env, policy, seeds, greedy=True):
    results = []
    for seed in seeds:
        _, metrics, total_r = run_episode(env, policy, seed, CONFIG, greedy=greedy)
        metrics["total_return"] = total_r
        results.append(metrics)
    return results


In [ ]:
MODEL_PATH = "dvrp_gnn_rl_final.pt"
SEEDS = CONFIG["test_seeds"]

policy, _ = load_policy(CONFIG, path=MODEL_PATH)
print(f"Modèle chargé : {MODEL_PATH}")
print(f"risk_lambda : {policy.risk_lambda.item()}\n")

def adjusted_service_rate(metrics_list):
    """
    service_rate = n_served / (n_clients_total - n_cancelled) : exclut du
    denominateur les clients annules (plafond structurel de l'environnement,
    hors controle de la policy), plutot que de les compter comme des clients
    manques au meme titre que les rejetes.
    """
    rates = []
    for m in metrics_list:
        n_playable = m["n_clients_total"] - m["n_cancelled"]
        rates.append(m["n_served"] / n_playable if n_playable > 0 else 0.0)
    return rates


def print_metrics_block(results, label, keys_extra):
    print(f"\n=== {label} ({len(results)} seeds) ===")
    print(f"{'Metric':<30s}  {'Mean':>10}  {'Std':>8}")
    print("-" * 52)

    adj_rates = adjusted_service_rate(results)
    print(f"{'service_rate (ajusté)':<30s}  {np.mean(adj_rates):>10.4f}  {np.std(adj_rates):>8.4f}")

    for k in keys_extra:
        vals = [m[k] for m in results]
        print(f"{k:<30s}  {np.mean(vals):>10.4f}  {np.std(vals):>8.4f}")

    return adj_rates


keys_extra = ["n_served", "n_cancelled", "n_rejected", "total_distance_km",
              "total_travel_time_min", "total_overtime_min", "total_lateness_min"]

env_gnn = VRPDynamicEnv()
gnn_results = evaluate(env_gnn, policy, SEEDS, greedy=True)
gnn_adj = print_metrics_block(gnn_results, "ÉVALUATION GNN", keys_extra)

env_nn = VRPDynamicEnv()
nn_results = [nearest_neighbor_policy(env_nn, seed) for seed in SEEDS]
nn_adj = print_metrics_block(nn_results, "NEAREST-NEIGHBOR", keys_extra)

env_ua = VRPDynamicEnv()
ua_results = [urgency_aware_policy(env_ua, seed) for seed in SEEDS]
ua_adj = print_metrics_block(ua_results, "URGENCE-AWARE (Solomon)", keys_extra)


In [ ]:
print(f"\n{'Metric':<30s}  {'GNN':>10}  {'NN':>10}  {'Urgence':>10}  {'Δ vs NN':>10}  {'Δ vs Urg':>10}")
print("-" * 96)

gnn_rate, nn_rate, ua_rate = np.mean(gnn_adj), np.mean(nn_adj), np.mean(ua_adj)
print(f"{'service_rate (ajusté)':<30s}  {gnn_rate:>10.4f}  {nn_rate:>10.4f}  {ua_rate:>10.4f}  "
      f"{gnn_rate-nn_rate:>+10.4f}  {gnn_rate-ua_rate:>+10.4f}")

for k in keys_extra:
    gnn_m = np.mean([m[k] for m in gnn_results])
    nn_m  = np.mean([m[k] for m in nn_results])
    ua_m  = np.mean([m[k] for m in ua_results])
    print(f"{k:<30s}  {gnn_m:>10.4f}  {nn_m:>10.4f}  {ua_m:>10.4f}  {gnn_m-nn_m:>+10.4f}  {gnn_m-ua_m:>+10.4f}")

gnn_km = np.mean([m["total_distance_km"] for m in gnn_results]) / np.mean([m["n_served"] for m in gnn_results])
nn_km  = np.mean([m["total_distance_km"] for m in nn_results])  / np.mean([m["n_served"] for m in nn_results])
ua_km  = np.mean([m["total_distance_km"] for m in ua_results])  / np.mean([m["n_served"] for m in ua_results])
print(f"\nkm/client -- GNN={gnn_km:.3f}  NN={nn_km:.3f}  Urgence={ua_km:.3f}")

decomp_df = decompose_reward(VRPDynamicEnv(), policy, SEEDS, CONFIG)
print(f"\n=== DÉCOMPOSITION DE LA REWARD ===")
print_reward_decomposition(decomp_df)
